In [ ]:
#Importacion de librerias
import requests
from sklearn.preprocessing import OrdinalEncoder
from imblearn.over_sampling import SMOTE
from sklearn.preprocessing import LabelEncoder
from collections import Counter
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer
from sklearn.impute import KNNImputer
from sqlalchemy import create_engine, text, inspect
from sqlalchemy.orm import sessionmaker
import pandas as pd
from unidecode import unidecode
import numpy as np
from tqdm import tqdm
import json
from pandas import json_normalize
import openpyxl
import concurrent.futures
from sklearn.datasets import make_classification
from sklearn.linear_model import LogisticRegression
from sklearn.tree import DecisionTreeClassifier
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import classification_report, confusion_matrix
import seaborn as sns
import matplotlib.pyplot as plt
from sklearn.model_selection import train_test_split, GridSearchCV, StratifiedShuffleSplit, StratifiedKFold
from sklearn.preprocessing import StandardScaler, OneHotEncoder
from sklearn.metrics import classification_report, confusion_matrix, f1_score, roc_auc_score, roc_curve, precision_recall_curve
import xgboost as xgb
import joblib  # Para guardar y cargar el modelo
import shap    # Para la interpretabilidad del modelo


In [ ]:
# Configura los detalles de la conexión desde variables de entorno.
# Copia .env.example a .env y define los valores. Nunca escribas credenciales aquí.
import os

host     = os.getenv('DB_HOST')
port     = os.getenv('DB_PORT', '1433')
database = os.getenv('DB_NAME')

if not all([host, database]):
    raise RuntimeError(
        "Faltan variables de entorno: define DB_HOST y DB_NAME. Ver .env.example"
    )

try:
    # Crea la cadena de conexión
    url = (
        'mssql+pyodbc://@{host}:{port}/{db}'
        '?trusted_connection=yes&driver=SQL+Server'
    ).format(host=host, port=port, db=database)

    # Crear conexion con base de datos
    engine = create_engine(url)
    print("Conexion a la base de datos realizada")

except Exception as e:
    print('Error:', e)


In [ ]:

with engine.connect() as connection:
    try:
        # 
        sql_query = text(''' 


SELECT 
    Identificacion,
    Periodo,
	Reintegro,
    Periodo_Siguiente,
    EX_Periodo_Siguiente_Normal,
    nuevo_periodo,
    Status,
	Tipo_Salto,
    Estado_Alumno,
    Tipo_estado_alumno,
    Modalidad,
    Programa,
    Semestre_SINU,
	ciclo,
	año,
    AÑO_MEN,
	Estado_Alumno,
    anio_grado,
    Genero,
    Estado_Pago,
    RANGO_EDAD,
    RANGO_SALARIO,
    ESTA_TRABAJANDO,
    METODO_FINANCIAMIENTO,
    ZONA_RESIDENCIA,
    REGIMEN_SISTEMA_SALUD,
    PERTENECE_GRUPO_ETNICO,
    GRUPO_ETNICO,
    TIENE_DISCAPACIDAD,
    DISCAPACIDAD,
    LGBTIQ,
	ciudad,
	localidad_normalizada
FROM academico.historial_academico


''')


    # CONVIERTE RESULTADO DE CONSULTA EN DATAFRAME    
        historial_academico = pd.read_sql_query(sql_query, engine)

    finally:
        
        connection.close()


In [ ]:

with engine.connect() as connection:
    try:
        # 
        sql_query = text(''' 


SELECT 
Identificacion, 
barrio, 
departamento, 
pais
FROM geo.ubicacion_estudiante


''')


    # CONVIERTE RESULTADO DE CONSULTA EN DATAFRAME    
        localización = pd.read_sql_query(sql_query, engine)

    finally:
        
        connection.close()


In [ ]:

with engine.connect() as connection:
    try:
        # 
        sql_query = text(''' 


SELECT
Periodo,
NOMBRE_GRUPO, 
IDENTIFICACION as Identificacion, 
NOMBRE_CONCEPTO, 
NOMBRE_CAUSA_NOTA
FROM becas.descuentos_beca


''')


    # CONVIERTE RESULTADO DE CONSULTA EN DATAFRAME    
        becas = pd.read_sql_query(sql_query, engine)

    finally:
        
        connection.close()


In [ ]:

with engine.connect() as connection:
    try:
        # 
        sql_query = text(''' 


select 
IDENTIFICACION as Identificacion, 
[MATERIAS INSCRITAS], 
[MATERIAS APROBADAS], 
COD_PERIODO as Periodo, 
Porcentaje_aprobacion
FROM academico.aprobacion_materias 


''')


    # CONVIERTE RESULTADO DE CONSULTA EN DATAFRAME    
        materias = pd.read_sql_query(sql_query, engine)

    finally:
        
        connection.close()


In [ ]:

with engine.connect() as connection:
    try:
        # 
        sql_query = text(''' 


select Periodo, Identificacion,TOTAL from 
financiera.cartera


''')


    # CONVIERTE RESULTADO DE CONSULTA EN DATAFRAME    
        finanza = pd.read_sql_query(sql_query, engine)

    finally:
        
        connection.close()


In [ ]:

with engine.connect() as connection:
    try:
        # 
        sql_query = text(''' 


select *
from plataforma.permanencia


''')


    # CONVIERTE RESULTADO DE CONSULTA EN DATAFRAME    
        permanencia = pd.read_sql_query(sql_query, engine)

    finally:
        
        connection.close()


In [ ]:

# Hago copia
df = historial_academico.copy()


In [ ]:

# Verifico Columnas y Filas
df.shape


In [ ]:

# Ahora haz el merge correctamente
df1 = df.merge(
    localización,
    on='Identificacion',
    how='left'
)


In [ ]:

# Verifico Columnas y Filas
df1.shape


In [ ]:

# Borro duplicados de identificación y periodo porque me interesa solammente cursado en ultimo periodo
becas = becas[~becas.duplicated(subset=['Identificacion', 'Periodo'], keep='last')]


In [ ]:

# Ahora haz el merge correctamente
df2 = df1.merge(
    becas,
    on=['Identificacion','Periodo'],
    how='left'
)


In [ ]:

# Verifico Columnas y Filas
df2.shape


In [ ]:

# Ahora haz el merge correctamente
df3 = df2.merge(
    materias,
    on=['Identificacion', 'Periodo'],
    how='left'
)


In [ ]:

# Verifico Columnas y Filas
df3.shape


In [ ]:

# Borro duplicados de identificación y periodo porque me interesa solammente cursado en ultimo periodo
finanza = finanza[~finanza.duplicated(subset=['Identificacion', 'Periodo'], keep='last')]


In [ ]:

# Ahora haz el merge correctamente
df4 = df3.merge(
    finanza,
    on=['Identificacion', 'Periodo'],
    how='left'
)


In [ ]:

# Verifico Columnas y Filas
df4.shape


In [ ]:

# Paso 0: Lista de periodos a eliminar
periodos_a_excluir = [
    '24I01', '24I02', '24I03', '24I04', '24I05', '24I06',
    '24I10', '24I11', '24I12', '24I13',
    '25I01', '25I02', '25I03', '25I04', '25I11', '25I12', '25I13'
]


In [ ]:

# Paso 1: Eliminar los periodos que no se desean
permanencia = permanencia[~permanencia['cod_periodo'].isin(periodos_a_excluir)]


In [ ]:

# Paso 2: Obtener el último periodo (cod_periodo) por cada estudiante
ultimo_periodo = permanencia.groupby('num_identificacion')['cod_periodo'].max().reset_index()
ultimo_periodo.rename(columns={'cod_periodo': 'ultimo_cod_periodo'}, inplace=True)


In [ ]:

# Paso 3: Unir esta información al DataFrame original
permanenciaa = permanencia.merge(ultimo_periodo, on='num_identificacion', how='left')


In [ ]:

# Paso 4: Filtrar solo las filas del último periodo por estudiante
permanenciaa = permanenciaa[permanenciaa['cod_periodo'] == permanenciaa['ultimo_cod_periodo']]


In [ ]:

# Paso 5: Eliminar duplicados (por si un curso se repite por error)
permanenciaa = permanenciaa.drop_duplicates(subset=['num_identificacion', 'curso', 'cod_periodo'])


In [ ]:

# Paso 6: Eliminar la columna auxiliar que ya no necesitamos
permanenciaa = permanenciaa.drop(columns=['ultimo_cod_periodo'])


In [ ]:

# Paso 7: Ordenar por estudiante y curso
permanenciaa = permanenciaa.sort_values(by=['num_identificacion', 'curso'])


In [ ]:

# Paso 8: Reiniciar índices
permanenciaa = permanenciaa.reset_index(drop=True)


In [ ]:

# Resultado final:
permanenciaa


In [ ]:

# Verifico Columnas
permanenciaa.columns


In [ ]:

# Renombro para que me cruzen las llaves
permanenciaa = permanenciaa.rename(columns={
    'num_identificacion': 'Identificacion',
    'cod_periodo': 'Periodo'
})


In [ ]:

# Verifico Columnas
permanenciaa.columns


In [ ]:

# Borro las variables que no necesito
permanenciaa = permanenciaa.drop(columns=[
    'SEMESTRE', 'NOM_UNIDAD', 'NOM_DEPENDENCIA', 'CRE_PROGRAMA',
    'CREDITOS_INSCRITOS', 'Materias_BU', 'Materias_B1', 'Materias_B2',
    'nivelado_en_materias', 'itemtype', 'itemmodule', 'itemname',
    'nombrecategoria', 'peso', 'finalgrade', 'estadoactividad',
    'estudiantes', 'curso', '#ASISTENCIAS'  
    
])


In [ ]:

# Verifico Columnas
permanenciaa.columns


In [ ]:

# Borro duplicados de identificación y periodo porque me interesa solammente cursado en ultimo periodo
permanenciaa = permanenciaa.drop_duplicates(subset=['Identificacion', 'Periodo'])


In [ ]:

# Verifico Columnas y Filas
df4.shape


In [ ]:

# Ahora haz el merge correctamente

df5 = df4.merge(
    permanenciaa,
    on=['Identificacion', 'Periodo'],
    how='left'
)


In [ ]:

# Verifico Columnas y Filas
df5.shape


In [ ]:
# DF final

df5


In [ ]:
# Verifico las columnas del DataFrame final

print(df5.columns.tolist())


In [ ]:
# Verifico las columnas del DataFrame final

print(df5.columns.tolist())

In [ ]:
df5.columns = df5.columns.str.strip().str.upper().str.replace(' ', '_')

In [ ]:
# Prototipo si quiero filtrar por Periodo y borro duplicados de identificación

#df5=df5.drop_duplicates(['Identificacion'])


#df5  = df5[df5['Periodo']=='2025B']

# Paso 0: Lista de periodos a eliminar
#año_a_excluir = [
#    '2024', '2025'
#]

# Paso 1: Eliminar los años que no se desean
#df5 = df5[~df5['AÑO'].isin(año_a_excluir)]

In [ ]:
## Verifico si hay valores nulos

df5.isnull().sum()


In [ ]:
# Verifico si hay valores nulos en la columna 'ASISTENCIA'
df5['ASISTENCIA'].value_counts(dropna=False)


In [ ]:
#Covversion de columnas sin espacios y mayusculas

df5.columns = df5.columns.str.strip().str.upper().str.replace(' ', '_')

In [ ]:
# ...imprime la lista para estar seguro de los nombres correctos
print(df5.columns.tolist())

In [ ]:
# Prototipo si quiero crear una columna de target de deserción

#df5['TARGET_DESERCION'] = df5['STATUS'].apply(
#    lambda x: 1 if str(x).strip().lower() == 'desercion' else 0
#)

In [ ]:
# Mi columna objetivo es 'TARGET_DESERCION' y ha sido creada para identificar si un estudiante ha desertado o no y convertido ha binario.


df5['TARGET_DESERCION'] = df5['STATUS'].apply(
    lambda x: 1 if str(x).strip().lower() == 'desercion' else 0
)

In [ ]:
# Columna objetivo
col_objetivo = 'TARGET_DESERCION'


In [ ]:
# Verifico si hay valores nulos

df5.isnull().sum()

In [ ]:
# =============================================================================
# PASO 1: CARGA Y LIMPIEZA DE DATOS
# =============================================================================
# ... (Aquí va todo tu código de conexión a la BD, extracción de las 6 tablas y merge hasta tener 'df5') ...
# Asumimos que 'df5' es el DataFrame resultante de todas tus fusiones.

# Estandarización de nombres de columnas
df5.columns = df5.columns.str.strip().str.upper().str.replace(' ', '_')
df5.rename(columns={'MATERIAS_INSCRITAS': 'MATERIAS_INSCRITAS', 'MATERIAS_APROBADAS': 'MATERIAS_APROBADAS'}, inplace=True, errors='ignore')


In [ ]:

# =============================================================================
# PASO 2: INGENIERÍA DE CARACTERÍSTICAS
# =============================================================================
print("\nAplicando ingeniería de características...")
df5['FLAG_NO_APROBO_NADA'] = np.where(df5['PORCENTAJE_APROBACION'] == 0, 1, 0)
df5['PORCENTAJE_APROBACION'] = df5['PORCENTAJE_APROBACION'].replace(0, np.nan)
print("Nueva característica 'FLAG_NO_APROBO_NADA' creada.")


In [ ]:
df5

In [ ]:

# =============================================================================
# PASO 3: SELECCIÓN DE CARACTERÍSTICAS Y PREPROCESAMIENTO
# =============================================================================

# Identificadores (no se usan como features, pero se conservan)
identificadores = ['IDENTIFICACION', 'PERIODO', 'AÑO']


In [ ]:

# Variable target
target = 'TARGET_DESERCION'


In [ ]:

columnas_numericas = [
    'SEMESTRE_SINU', 'MATERIAS_INSCRITAS', 'MATERIAS_APROBADAS',
    'PORCENTAJE_APROBACION', 'TOTAL', 'FLAG_NO_APROBO_NADA'
]


In [ ]:

columnas_categoricas = [
    'TIPO_SALTO', 'MODALIDAD', 'GENERO', 'ESTADO_PAGO', 'RANGO_EDAD',
    'RANGO_SALARIO', 'ESTA_TRABAJANDO', 'METODO_FINANCIAMIENTO',
    'ZONA_RESIDENCIA', 'REGIMEN_SISTEMA_SALUD', 'ASISTENCIA'
]


In [ ]:

# Variable target
target = 'TARGET_DESERCION'

features = columnas_numericas + columnas_categoricas


In [ ]:

# Seleccionar solo identificadores + features + target
columnas_ml = identificadores + features + [target]


In [ ]:

# Crear un DataFrame reducido para ML
df_ml = df5[columnas_ml]


In [ ]:

# Transformaciones
numeric_transformer = Pipeline(steps=[
    ('imputer', SimpleImputer(strategy='median')),
    ('scaler', StandardScaler())
])

categorical_transformer = Pipeline(steps=[
    ('imputer', SimpleImputer(strategy='most_frequent'))
    # sin OneHotEncoder para mantener columnas iguales
])

# Aplicar imputación a numéricas
df_num = pd.DataFrame(
    numeric_transformer.fit_transform(df5[columnas_numericas]),
    columns=columnas_numericas,
    index=df5.index
)

# Aplicar imputación a categóricas
df_cat = pd.DataFrame(
    categorical_transformer.fit_transform(df5[columnas_categoricas]),
    columns=columnas_categoricas,
    index=df5.index
)

# Reconstruir el DataFrame final con misma forma
df_final_ml = pd.concat([
    df5[identificadores],   # IDENTIFICACION y AÑO originales
    df_num,                 # numéricas imputadas
    df_cat,                 # categóricas imputadas
    df5[[target]]           # target intacto
], axis=1
)

In [ ]:
# Comparar identificadores originales vs procesados
comparacion = (df5[identificadores].reset_index(drop=True) == 
               df_final_ml[identificadores].reset_index(drop=True))

# Mostrar si todas las filas son iguales
print("¿IDENTIFICACION coincide en todas las filas?:", comparacion['IDENTIFICACION'].all())
print("¿AÑO coincide en todas las filas?:", comparacion['AÑO'].all())

# Si quieres ver las primeras diferencias (si existieran)
diferencias = df5[identificadores].reset_index(drop=True).compare(
    df_final_ml[identificadores].reset_index(drop=True)
)
print("Diferencias encontradas:")
print(diferencias.head())

In [ ]:
df_final_ml

In [ ]:
df_final_ml['AÑO'].value_counts()

In [ ]:
df_final_ml.shape

In [ ]:
df_ml.shape

In [ ]:
df_ml

In [ ]:

# Guardar en Excel
#df_final_ml.to_excel("Despliegue2024.xlsx", index=Falseghnt)  # index=False para no guardar la columna de índices

In [ ]:
df_desercion = df_final_ml.copy()

In [ ]:
df_desercion.columns

In [ ]:
df_desercion.shape

In [ ]:
#df_desercion = df_desercion[~df_desercion['AÑO'].astype(str).isin(['2024', '2025'])]

In [ ]:
df_desercion.shape

### entrnamiebto


In [ ]:
X.columns

In [ ]:
#PROTOTIPOI
# =============================================================================
# PASO 4: DIVISIÓN DE DATOS
# =============================================================================
#X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.20, random_state=42, stratify=y)
#print(f"\nDatos divididos: {len(X_train)} para entrenamiento, {len(X_test)} para prueba.")


# División temporal: entrenar hasta 2023, validar en 2024
df_train = df_desercion[df_desercion['AÑO'] <= 2023]
df_test  = df_desercion[df_desercion['AÑO'] == 2024]

X_train, y_train = df_train.drop(columns=identificadores + [target]), df_train[target]
X_test, y_test   = df_test.drop(columns=identificadores + [target]), df_test[target]

print(f"\nDatos divididos: {len(X_train)} para entrenamiento (<=2023), {len(X_test)} para prueba (2024).")


In [ ]:
X

In [ ]:
from sklearn.preprocessing import OrdinalEncoder, StandardScaler
from sklearn.compose import ColumnTransformer

# === 1) Preprocesador SIN expandir columnas (mantiene 21 columnas totales) ===
preprocessor = ColumnTransformer(
    transformers=[
        ('num', StandardScaler(), columnas_numericas),
        ('cat', OrdinalEncoder(handle_unknown='use_encoded_value', unknown_value=-1), columnas_categoricas)
    ],
    remainder='drop'
)

# === 2) Transformar TODO X (todas las filas/columnas excepto IDs y target) ===
X_transformado = preprocessor.fit_transform(X)

# === 3) Reconstruir DataFrame con los nombres correctos ===
nombres_finales = columnas_numericas + columnas_categoricas
df_binarizado = pd.DataFrame(X_transformado, columns=nombres_finales, index=X.index)

# (opcional) Dejar categóricas como enteros en vez de float
df_binarizado[columnas_categoricas] = df_binarizado[columnas_categoricas].round().astype('Int64')

# === 4) Identificadores y Target (del df_desercion para no perderlos) ===
df_identificadores = df_desercion[identificadores].reset_index(drop=True)
df_binarizado = df_binarizado.reset_index(drop=True)
df_target = df_desercion[[target]].reset_index(drop=True)

# === 5) Concatenar: IDs + features (num+cat) + TARGET ===
df_final_completo = pd.concat([df_identificadores, df_binarizado, df_target], axis=1)

# === 6) Orden EXACTO de columnas para que sea igual a df_ml (21 columnas) ===
orden_esperado = identificadores + columnas_numericas + columnas_categoricas + [target]
df_final_completo = df_final_completo[orden_esperado]

# === 7) Validaciones rápidas ===
print("Shape df_final_completo:", df_final_completo.shape)  # <-- debería ser (924831, 21)
print("Columnas correctas:", df_final_completo.columns.tolist()==orden_esperado)

df_final_completo

In [ ]:
# Guardar en Excel
df_final_completo.to_excel("Despliegue2024y2025.xlsx", index=False)  # index=False para no guardar la columna de índices

In [ ]:
dff = df_final_completo[~df_final_completo['AÑO'].astype(str).isin(['2024', '2025'])]

In [ ]:

# df es tu DataFrame
columnas_a_borrar = ['IDENTIFICACION', 'PERIODO']
dff = dff.drop(columns=columnas_a_borrar)

In [ ]:
dff

In [ ]:
# ==========================
# 1. Librerías
# ==========================
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split, StratifiedKFold
from sklearn.preprocessing import StandardScaler, OrdinalEncoder
from sklearn.compose import ColumnTransformer
from imblearn.pipeline import Pipeline as ImbPipeline
from imblearn.over_sampling import SMOTE
from sklearn.metrics import f1_score, recall_score, precision_score, roc_auc_score, classification_report, confusion_matrix
from sklearn.ensemble import StackingClassifier
from sklearn.linear_model import LogisticRegression
from xgboost import XGBClassifier
from lightgbm import LGBMClassifier
from skopt import BayesSearchCV
import shap
import matplotlib.pyplot as plt
import seaborn as sns
import joblib


In [ ]:

# ==========================
# 2. Datos
# ==========================
target = 'TARGET_DESERCION'

X = dff.drop(columns=[target])
y = dff[target]

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.3, random_state=42, stratify=y
)

num_cols = X_train.select_dtypes(include=['int64','float64']).columns.tolist()
cat_cols = X_train.select_dtypes(exclude=['int64','float64']).columns.tolist()


In [ ]:

# ==========================
# 3. Preprocesamiento
# ==========================
preprocessor = ColumnTransformer([
    ('num', StandardScaler(), num_cols),
    ('cat', OrdinalEncoder(handle_unknown='use_encoded_value', unknown_value=-1), cat_cols)
])


In [ ]:

# ==========================
# 4. Modelos base y hiperparámetros Bayesianos
# ==========================
estimadores = {
    "XGB": XGBClassifier(use_label_encoder=False, eval_metric='logloss', random_state=42),
    "LGBM": LGBMClassifier(random_state=42)
}

param_bayes = {
    "XGB": {
        'clf__n_estimators': (100, 500),
        'clf__max_depth': (3, 10),
        'clf__learning_rate': (0.01,0.3,'log-uniform'),
        'clf__subsample': (0.6,1.0),
        'clf__colsample_bytree': (0.6,1.0)
    },
    "LGBM": {
        'clf__n_estimators': (100,500),
        'clf__num_leaves': (31,150),
        'clf__learning_rate': (0.01,0.3,'log-uniform'),
        'clf__subsample': (0.6,1.0)
    }
}


In [ ]:

mejores_modelos = {}
for nombre, modelo in estimadores.items():
    print(f"\n🔍 Optimización Bayesiana {nombre}...")
    pipe = ImbPipeline([
        ('preprocessor', preprocessor),
        ('smote', SMOTE(random_state=42)),
        ('clf', modelo)
    ])
    bayes = BayesSearchCV(
        estimator=pipe,
        search_spaces=param_bayes[nombre],
        n_iter=30,
        scoring='f1',
        cv=StratifiedKFold(n_splits=3),
        random_state=42,
        n_jobs=-1,
        verbose=1
    )
    bayes.fit(X_train, y_train)
    print(f"✅ Mejor F1 {nombre}: {bayes.best_score_:.4f}")
    print(f"Mejores parámetros: {bayes.best_params_}")
    mejores_modelos[nombre] = bayes.best_estimator_


In [ ]:

# ==========================
# 5. Stacking avanzado
# ==========================
stacking = StackingClassifier(
    estimators=[(k,v.named_steps['clf']) for k,v in mejores_modelos.items()],
    final_estimator=LogisticRegression(max_iter=1000, random_state=42),
    cv=StratifiedKFold(n_splits=3),
    n_jobs=-1,
    passthrough=True
)

pipeline_final = ImbPipeline([
    ('preprocessor', preprocessor),
    ('smote', SMOTE(random_state=42)),
    ('clf', stacking)
])

pipeline_final.fit(X_train, y_train)


In [ ]:

# ==========================
# 6. Umbral óptimo F1/Recall desertores
# ==========================
y_proba_test = pipeline_final.predict_proba(X_test)[:,1]
prec, rec, thr = precision_recall_curve(y_test, y_proba_test)
f1_scores = 2*prec*rec/(prec+rec+1e-6)
best_idx = np.argmax(f1_scores)
best_thr = thr[best_idx] if best_idx < len(thr) else 0.5
y_pred_best = (y_proba_test >= best_thr).astype(int)

print("\n✅ Métricas finales con umbral optimizado")
print("F1:", f1_score(y_test, y_pred_best))
print("Recall:", recall_score(y_test, y_pred_best))
print("Precision:", precision_score(y_test, y_pred_best))
print("AUC:", roc_auc_score(y_test, y_proba_test))
print(classification_report(y_test, y_pred_best))


In [ ]:

# ==========================
# 7. SHAP Top 20
# ==========================
explainer_final = shap.TreeExplainer(pipeline_final.named_steps['clf'].estimators_[0])
X_proc_final = pipeline_final.named_steps['preprocessor'].transform(X_train)
shap_values_final = explainer_final.shap_values(X_proc_final)
shap.summary_plot(shap_values_final, X_proc_final, feature_names=num_cols+cat_cols, max_display=20)


In [ ]:


# ==========================
# 8. Guardar modelo
# ==========================

# ✅ LÍNEA CLAVE: Añade el umbral como un atributo al pipeline
pipeline_final.best_threshold = best_thr 

# Ahora guarda el pipeline, que ya contiene el umbral
joblib.dump(pipeline_final, "modelo_ultrapotente_v2.pkl") 
print(f"✅ Modelo guardado con umbral: {pipeline_final.best_threshold:.4f}")


In [ ]:

# ==========================
# 9. Función de predicción con umbral
# ==========================
def predecir_nuevo(df_nuevo, umbral=best_thr):
    proba = pipeline_final.predict_proba(df_nuevo)[:,1]
    pred = (proba >= umbral).astype(int)
    return pred, proba
